<a href="https://colab.research.google.com/github/martingkc/NLP_Notebooks/blob/main/CLIP_Image_Encoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install evaluate pandas rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.3 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=10bc62fc0e79c0715dc645f270b5a34172a3ac301fa67191774c1c00b8442b7c
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
epochs = 100


class PatchEmbedding(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_channels=1, embed_dim=768):
        super().__init__()
        self.patch_size = patch_size
        self.unfold = nn.Unfold(kernel_size=patch_size, stride=patch_size)
        self.patch_embed = nn.Linear(in_channels * patch_size ** 2, embed_dim)
        self.img_size = img_size
        self.embed_dim = embed_dim

    def forward(self, x):
        """
        Takes a [Batch_Size, 1, 224, 224] tensor and splits it into 16x16 chunks which then get flattened into tensors 196x1
        which get projected on our embedding dim of 768 resulting in a tensor shaped: [B, 196, 768]
        """
        patches = self.unfold(x)
        patches = patches.transpose(1, 2)
        return self.patch_embed(patches)

## Implementation of 2D-Rotarional Position Encoding
Implementation taken from this repo: [s-chh/2D-Positional-Encoding-Vision-Transformer](https://github.com/s-chh/2D-Positional-Encoding-Vision-Transformer/tree/main)

In [ ]:
# X-axis specific values
def get_x_positions(n_patches, start_idx=0):
    n_patches_ = int(n_patches ** 0.5)                                    # Number of patches along 1 dimension

    x_positions = torch.arange(start_idx, n_patches_ + start_idx)         # N_
    x_positions = x_positions.unsqueeze(0)                                # 1, N_
    x_positions = torch.repeat_interleave(x_positions, n_patches_, 0)     # N_ , N_                         Matrix to replicate positions of patches on x-axis
    x_positions = x_positions.reshape(-1)                                 # N_ , N_  ->  N_ ** 2  =  N

    return x_positions


# Y-axis specific values
def get_y_positions(n_patches, start_idx=0):
    n_patches_ = int(n_patches ** 0.5)                                    # Number of patches along 1 dimension

    y_positions = torch.arange(start_idx, n_patches_+start_idx)           # N_
    y_positions = torch.repeat_interleave(y_positions, n_patches_, 0)     # N_ , N_  ->  N_ ** 2  =  N                  Matrix to replicate positions of patches on y-axis

    return y_positions

class RotatoryPositionEmbedding2D(nn.Module):
    def __init__(self, seq_len, embed_dim):
        super().__init__()
        self.embed_dim = embed_dim // 2                                           # Split the dimensions into two parts. We will use 1 part for x-axis and the other part for y-axis

        n_patches = seq_len - 1

        x_positions  = get_x_positions(n_patches, start_idx=1).reshape(-1, 1)     # N  ->  N, 1
        x_sin, x_cos = self.generate_rope1D(x_positions)                          # 1, 1, N, E//2    ,    1, 1, N, E//2
        self.register_buffer("x_cos", x_cos)                                      # Register_buffer for easy switching of device
        self.register_buffer("x_sin", x_sin)                                      # Register_buffer for easy switching of device

        y_positions  = get_y_positions(n_patches, start_idx=1).reshape(-1, 1)     # N  ->  N, 1
        y_sin, y_cos = self.generate_rope1D(y_positions)                          # 1, 1, N, E//2    ,    1, 1, N, E//2
        self.register_buffer("y_cos", y_cos)                                      # Register_buffer for easy switching of device
        self.register_buffer("y_sin", y_sin)                                      # Register_buffer for easy switching of device


    def generate_rope1D(self, sequence):
        '''
        Create theta as per the equation in the RoPe paper: theta = 10000 ^ -2(i-1)/d for i belongs to [1, 2, ... d/2].
        Note this d/2 is different from previous x/y axis split.
        '''
        sequence   = F.pad(sequence, (0, 0, 1, 0))                                              # N, 1        ->  N + 1, 1 = S      Pad with 0 to account for classification token
        thetas     = -2 * torch.arange(start=1, end=self.embed_dim//2+1) / self.embed_dim       # E//4
        thetas     = torch.repeat_interleave(thetas, 2, 0)                                      # E//2
        thetas     = torch.pow(10000, thetas)                                                   # E//2
        values     = sequence * thetas                                                          # S, 1 * E//2 -> S, E//2
        cos_values = torch.cos(values).unsqueeze(0).unsqueeze(0)                                # S, E//2     -> 1, 1, S, E//2      Precompute and store cos values
        sin_values = torch.sin(values).unsqueeze(0).unsqueeze(0)                                # S, E//2     -> 1, 1, S, E//2      Precompute and store sin values
        return sin_values, cos_values


    def forward(self, x):
        x_x = x[:, :, :, :self.embed_dim]                                            # B, H, S, E//2                                            Split half of the embeddings of x for x-axis
        x_y = x[:, :, :, self.embed_dim:]                                            # B, H, S, E//2                                            Split half of the embeddings of x for y-axis

        x_x1 = x_x * self.x_cos                                                      # B, H, S, E//2  *  1, 1, S, E//2   ->  B, H, S, E//2      Multiply x-axis part of input with its cos factor as per the eq in RoPe
        x_x_shifted = torch.stack((-x_x[:, :, :, 1::2], x_x[:, :, :, ::2]), -1)      # B, H, S, E//2                     ->  B, H, S, E//4, 2   Shift values for sin multiplications are per the eq in RoPe
        x_x_shifted = x_x_shifted.reshape(x_x.shape)                                 # B, H, S, E//4, 2                  ->  B, H, S, E//2
        x_x2 = x_x_shifted * self.x_sin                                              # B, H, S, E//2  *  1, 1, S, E//2   ->  B, S, E//2         Multiply x-axis part of x with its sin factor as per the eq in RoPe
        x_x = x_x1 + x_x2                                                            # Add sin and cosine value

        x_y1 = x_y * self.y_cos                                                      # B, H, S, E//2  *  1, 1, S, E//2   ->  B, H, S, E//2      Multiply y-axis part of input with its cos factor as per the eq in RoPe
        x_y_shifted = torch.stack((-x_y[:, :, :, 1::2], x_y[:, :, :, ::2]), -1)      # B, H, S, E//2                     ->  B, H, S, E//4, 2   Shift values for sin multiplications are per the eq in RoPe
        x_y_shifted = x_y_shifted.reshape(x_y.shape)                                 # B, H, S, E//4, 2                  ->  B, H, S, E//2
        x_y2 = x_y_shifted * self.y_sin                                              # B, H, S, E//2  *  1, 1, S, E//2   ->  B, H, S, E//2      Multiply y-axis part of x with its sin factor as per the eq in RoPe
        x_y = x_y1 + x_y2                                                            # Add sin and cosine value

        x = torch.cat((x_x, x_y), -1)                                                # B, H, S, E//2  cat  B, H, S, E//2 -> B, H, S, E          Combine x and y rotational projections
        return x



# MultiHeadAttention and Encoder-Only Transformer Block
We implement the version indicated on the ViT paper also used on the ViT tutorial, using 12 heads.


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, seq_len):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.qkv = nn.Linear(embed_dim, 3 * embed_dim)
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.rope = RotatoryPositionEmbedding2D(seq_len, self.head_dim)

    def forward(self, x):
        B, S, E = x.shape # [batch_size, 197, 768]
        qkv = self.qkv(x) # [batch_size, 197, 2304]
        qkv = qkv.reshape(B, S, 3, self.num_heads, self.head_dim) # reshapes to [batch_size, 197, 3, 12, 64]
        q, k, v = qkv.unbind(2) # splits into [batch_size, 197, 12, 64]
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        q = self.rope(q)
        k = self.rope(k)
        attn = (q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5) # dot prod between every key and query. resulting in [B, 12, 197, 197]
        attn = attn.softmax(-1)
        x = (attn @ v).transpose(1, 2).reshape(B, S, E)
        return self.proj(x)

class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_dim, seq_len):
        super().__init__()
        self.attn = MultiHeadAttention(embed_dim, num_heads, seq_len)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim),
            nn.GELU(),
            nn.Linear(mlp_dim, embed_dim)
        )
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = x + self.attn(self.norm1(x)) # normalize and run attention + residual connection
        x = x + self.mlp(self.norm2(x)) # normalize and transform each patch + residual
        return x

# The ViT
The ViT we're going to use in this example will have a depth of 6. Its sequence length is going to be equal to our patch number + 1 for the CLS token.

In [ ]:
class VisionTransformer(nn.Module):
    def __init__(self, img_size=224, patch_size=16, num_classes=10, embed_dim=512, num_heads=8, depth=6, mlp_dim=1024, channels = 1):
        super().__init__()
        seq_len = (img_size // patch_size) ** 2 + 1 # Our seq len is going to be the num of patches + one CLS token
        self.patch_embedding = PatchEmbedding(img_size, patch_size, channels , embed_dim) # patch the image
        self.transformer_blocks = nn.ModuleList([
            TransformerEncoderBlock(embed_dim, num_heads, mlp_dim, seq_len) for _ in range(depth)
        ]) # Our ViT is going to have a depth of 6.
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim)) # Assign the CLS token
        self.proj = nn.Linear(embed_dim, embed_dim)
    def forward(self, x):
        B = x.size(0) # batch size
        x = self.patch_embedding(x)
        cls_tokens = self.cls_token.expand(B, -1, -1) # cls token is shaped [1, 1, 512] this expands it to [B, 1, 512]
        x = torch.cat((cls_tokens, x), dim=1) # concatenate the cls token
        for block in self.transformer_blocks:
            x = block(x) # run the input through the 6 transformer blocks
        return self.proj(x[:, 0]) # spit out the embedding

# The Dataset
The dataset that I will use for this example is one that I had used previously for a uni challenge which is composed of B/W chest x-rays, a list of labels of the various pathologies and image and textual embeddings obtained through a pre trained CLIP model. The dataset is quite small so I suggest using another one.

In [ ]:
import torch.optim as optim
from datasets import load_dataset
import pandas as pd
from torch.utils.data import Dataset


dataset = load_dataset("Martingkc/MediBert_Dataset")
df = pd.DataFrame(dataset['train'])
df.info()



README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/480M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/485M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2380 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2380 [00:00<?, ? examples/s]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2380 entries, 0 to 2379
Data columns (total 21 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Image Index         2380 non-null   object
 1   Texts               2380 non-null   object
 2   View Position       2380 non-null   object
 3   Image Features      2380 non-null   object
 4   Text Features       2380 non-null   object
 5   Atelectasis         2380 non-null   int64 
 6   Cardiomegaly        2380 non-null   int64 
 7   Effusion            2380 non-null   int64 
 8   Infiltration        2380 non-null   int64 
 9   Mass                2380 non-null   int64 
 10  Nodule              2380 non-null   int64 
 11  Pneumonia           2380 non-null   int64 
 12  Pneumothorax        2380 non-null   int64 
 13  Consolidation       2380 non-null   int64 
 14  Edema               2380 non-null   int64 
 15  Emphysema           2380 non-null   int64 
 16  Fibrosis            2380

In [ ]:

from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df, test_size=0.2)


In [ ]:
from transformers import CLIPTokenizer, CLIPTextModel

# Uncomment if you're using a dataset without embeddings of the text.
#tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
#text_model = CLIPTextModel.from_pretrained("openai/clip-vit-base-patch32").cuda()

def get_embedding(caption):
    inputs = tokenizer(caption, return_tensors="pt", padding=True, truncation=True).to("cuda")
    with torch.no_grad():
        return text_model(**inputs).pooler_output.squeeze(0).cpu()

class DatasetPrepper(Dataset):
  """
  This is the dataset loader that we will use to "Normalize" our data.
  It basically ensures that the images are greyscale and that they are of size 224x224.
  If the text embeddings are missing they can be generated using the get_embedding function.
  """
  def __init__(self, df, embed_text=False, embed_model = None,  transform=None):
        self.df = df.reset_index(drop=True)
        self.embed_text = embed_text
        self.embed_model = embed_model
        self.transform = transform or transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.Grayscale(num_output_channels=1),
            transforms.ToTensor(),
        ])

  def __len__(self):
        return len(self.df)

  def __getitem__(self, idx):
          row = self.df.iloc[idx]
          img = self.transform(row['Image'])
          if not self.embed_text:
            text_emb = torch.tensor(row['Text Features'], dtype=torch.float32)
          else:
            text_emb = get_embedding(row['Texts'])
          text_label = row['Texts']
          return img, text_emb, text_label

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_projection.weight               

# Contrastive Learning with Info NCE Loss


**Similarity matrix**

$$S_{ij} = \frac{v_i^\top u_j}{\tau \|v_i\| \|u_j\|}$$

**Image → text loss**

$$\mathcal{L}_{\text{i2t}} = -\frac{1}{N} \sum_{i=1}^{N} \log \frac{\exp(S_{ii})}{\sum_{j=1}^{N} \exp(S_{ij})}$$

**Text → image loss**

$$\mathcal{L}_{\text{t2i}} = -\frac{1}{N} \sum_{i=1}^{N} \log \frac{\exp(S_{ii})}{\sum_{j=1}^{N} \exp(S_{ji})}$$

**Final loss**

$$\mathcal{L}_{\text{InfoNCE}} = \frac{\mathcal{L}_{\text{i2t}} + \mathcal{L}_{\text{t2i}}}{2}$$

Where $v_i$ = image embedding, $u_j$ = text embedding, $\tau$ = temperature (0.07),

In [ ]:
from torch.utils.data import DataLoader
import torch.nn as nn

def info_nce_loss(image_emb, text_emb, temperature=0.07):
    image_emb = F.normalize(image_emb, dim=-1)
    text_emb = F.normalize(text_emb, dim=-1)

    # similarity matrix [N, N]
    logits = (image_emb @ text_emb.T) / temperature

    # labels are just 0,1,2...N-1 — the diagonal is always correct
    labels = torch.arange(len(logits), device=logits.device)

    # cross entropy in both directions
    loss_i = F.cross_entropy(logits, labels)        # image -> text
    loss_t = F.cross_entropy(logits.T, labels)      # text -> image

    return (loss_i + loss_t) / 2

model = VisionTransformer(embed_dim=512).cuda()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
train_dataset = DatasetPrepper(train_df)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

for i in range(epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels, _ in train_loader:  # Iterate over train_loader
        inputs, labels = inputs.cuda(), labels.cuda()
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = info_nce_loss(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{i+1}/{epochs}], Loss: {running_loss/len(train_loader)}")

Epoch [1/100], Loss: 3.455480194091797
Epoch [2/100], Loss: 3.3265002727508546
Epoch [3/100], Loss: 3.2535940726598103
Epoch [4/100], Loss: 3.140743672847748
Epoch [5/100], Loss: 3.0916235287984213
Epoch [6/100], Loss: 3.040103089809418
Epoch [7/100], Loss: 3.0355929215749105
Epoch [8/100], Loss: 2.9835100293159487
Epoch [9/100], Loss: 2.9211466749509176
Epoch [10/100], Loss: 2.9099875609079997
Epoch [11/100], Loss: 2.8762012481689454
Epoch [12/100], Loss: 2.82938179175059
Epoch [13/100], Loss: 2.80175465742747
Epoch [14/100], Loss: 2.7691750526428223
Epoch [15/100], Loss: 2.749283854166667
Epoch [16/100], Loss: 2.660605323314667
Epoch [17/100], Loss: 2.5850022236506143
Epoch [18/100], Loss: 2.5406399766604104
Epoch [19/100], Loss: 2.4774505496025085
Epoch [20/100], Loss: 2.3686286131540935
Epoch [21/100], Loss: 2.3032429655392965
Epoch [22/100], Loss: 2.1554847101370496
Epoch [23/100], Loss: 2.0325897932052612
Epoch [24/100], Loss: 1.8804215749104818
Epoch [25/100], Loss: 1.7183127085

In [ ]:
label_df = test_df.drop_duplicates(subset=['Texts']).reset_index(drop=True)


In [ ]:
import torch
import numpy as np
import random
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score
def cosine_similarity(a, b):
    a = F.normalize(a, dim=-1)
    b = F.normalize(b, dim=-1)
    return (a * b).sum(dim=-1)

def eval_model(prediction_embs, true_labels):
    embs = label_df["Text Features"].values
    text = label_df["Texts"].values
    results = []

    for e, l in zip(prediction_embs, true_labels):
        similarity = torch.stack([
            cosine_similarity(e, torch.tensor(embs[i]))
            for i in range(len(embs))
        ])
        pred_label = text[torch.argmax(similarity)]
        results.append(pred_label == l)

    return {
        "accuracy": np.mean(results),
        "total": len(results),
        "correct": sum(results)
    }

def eval_recall_at_k(prediction_embs, true_text_embs, k=10):
    prediction_embs = F.normalize(prediction_embs, dim=-1)
    true_text_embs = F.normalize(true_text_embs, dim=-1)

    correct = 0
    n = len(prediction_embs)

    for i in range(n):
        # pick 9 random indices that are NOT the correct one
        wrong_indices = random.sample([j for j in range(n) if j != i], k-1)
        candidate_indices = wrong_indices + [i]  # inject correct at the end

        candidates = true_text_embs[candidate_indices]  # [10, D]
        sim = prediction_embs[i] @ candidates.T         # [10]

        if sim.argmax().item() == k-1:  # correct is always at index k-1
            correct += 1

    return {f"accuracy@{k}": correct / n, "correct": correct, "total": n}



In [ ]:
from sklearn.model_selection import train_test_split


# Create test dataset and dataloader
test_dataset = DatasetPrepper(test_df)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
all_predictions = []
all_true_text_embs = []
all_true_texts = []

model.eval()
with torch.no_grad():
    for inputs, labels, texts in test_loader:
        inputs, labels = inputs.cuda(), labels.cuda()
        outputs = model(inputs)
        all_predictions.append(outputs.cpu())
        all_true_text_embs.append(labels.cpu())
        all_true_texts.extend(texts)

predictions_tensor = torch.cat(all_predictions)
true_labels_tensor = torch.cat(all_true_text_embs)

metrics = eval_recall_at_k(predictions_tensor, true_labels_tensor, k=5)
print(f"accuracy over 5 unique labels:{metrics}")

metrics = eval_recall_at_k(predictions_tensor, true_labels_tensor, k=10)
print(f"accuracy over 10 unique labels:{metrics}")

metrics = eval_recall_at_k(predictions_tensor, true_labels_tensor, k=15)
print(f"accuracy over 15 unique labels:{metrics}")

metrics = eval_model(predictions_tensor, all_true_texts)
print("Evaluation Metrics over 290 unique labels:")
for metric_name, metric_value in metrics.items():
    print(f"{metric_name}: {metric_value}")




recall@5:{'accuracy@5': 0.31092436974789917, 'correct': 148, 'total': 476}
recall@10:{'accuracy@10': 0.16806722689075632, 'correct': 80, 'total': 476}
recall@15:{'accuracy@15': 0.14705882352941177, 'correct': 70, 'total': 476}
Evaluation Metrics:
accuracy: 0.029411764705882353
total: 476
correct: 14
